# EDA 13 — Mechanism × Yee Class-3 correspondence, reported as **proportions**

**Purpose.** EDA 12 (v3) established the leaf-level correspondence between the EDA 9 mechanism tree and
Yee & Dennett's (2022) attribute labels, but reported it in *lift* units ("enriched, lift 2.32 and 3.37";
"significantly depleted"). Lift is a ratio of two conditional probabilities and is not self-interpreting
for a reader: a lift of 3.37 says nothing about whether the underlying rate moved from 1 % to 3 % or from
25 % to 84 %. This notebook **re-reports the identical tests in proportion units** so the main-chapter prose
can say what actually happened:

> Super-gentrification occurred in **X %** of outflow-internal-majority MSOAs, compared with **Y %** of all
> London MSOAs.

instead of

> ~~Super-gentrification was enriched in the outflow-internal-majority leaf, with lift values of 2.32 and 3.37.~~

**Nothing about the analysis changes.** Same tree constants (`INFLOW_MIN = 0.25`, `EXT_MAJ = 0.50`), same Yee
aggregation, same Fisher exact tests, same Benjamini–Hochberg correction. What changes is the *reporting
layer*: proportions and percentage-point gaps drive the prose; ratios, odds ratios, p and q move to an
appendix table (§4, exported as `eda13_appendix_ratios_*.csv`). A consistency check in §5 asserts that the
ratios recovered here reproduce the EDA 12 lift values to two decimals, so the two notebooks cannot drift.

**Reporting conventions adopted here**

| EDA 12 phrasing | EDA 13 phrasing |
|---|---|
| "enriched, lift 2.32" | "occurred in X % of leaf MSOAs vs Y % of all London MSOAs (+Z pp)" |
| "significantly depleted" | "occurred less frequently in the leaf than in the London sample overall" |
| "lift 0.43, p = 0.0055" | "12 % vs 21 % (−9 pp); the difference is unlikely under independence (appendix)" |

**Temporal framing (unchanged from EDA 12 v2/v3).** Yee's labels describe 2001–2011 attribute change, so
**2011 = concurrent validation** and **2021 = decade-later persistence test**. The 2021 rows are never read
as validating 2021 flows against 2011-era labels.

**Inputs:** `eda4_results_for_phase3_20260626.csv`, `msoa_cascade_national_frame_20260625.csv`,
`yee_LSOA_labels_forMapping.csv`, `lsoa11_to_msoa11.csv`, `MSOA_2011_to_2021_lookup_for_identification.csv`.

**Outputs:** `eda13_proportions_main_20260723.csv`, `eda13_appendix_ratios_20260723.csv`,
`eda13_chapter_prose_20260723.md`, `fig_eda13_proportions_by_leaf.png`.

In [ ]:
# ── setup, inputs, EDA 9 mechanism tree (inline, identical to EDA 12 v3) ─────
import sys
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
from pyprojroot import here
import matplotlib.pyplot as plt

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'
OUT_DIR    = ROOT / 'outputs'
FIG_DIR    = OUT_DIR / 'comparison_figs'
NATF       = OUT_DIR / 'msoa_cascade_national_frame_20260625.csv'
EDA4       = OUT_DIR / 'eda4_results_for_phase3_20260626.csv'
YEE_LABELS = DATA_DIR / 'yee_LSOA_labels_forMapping.csv'
LOOKUP     = OUT_DIR / 'lsoa11_to_msoa11.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

STAMP = '20260723'

INFLOW_MIN, EXT_MAJ = 0.25, 0.50   # EDA 9: empirically gapped / stated majority convention
LEAVES = ['inflow-driven', 'frame-sensitive', 'outflow-external', 'outflow-internal']

df = pd.read_csv(EDA4).merge(
    pd.read_csv(NATF)[['msoa11cd', 'Ext_Outflow_nat_11', 'Outflow_Poorer_nat_11',
                       'Ext_Outflow_nat_21', 'Outflow_Poorer_nat_21']], on='msoa11cd')

# MSOA names (e.g. 'Camden 026') so rosters are readable against fig 25 / EDA 10-11 case labels
NAMES = next(p for p in (DATA_DIR / 'MSOA_2011_to_2021_lookup_for_identification.csv',
                         OUT_DIR  / 'MSOA_2011_to_2021_lookup_for_identification.csv') if p.exists())
nm = (pd.read_csv(NAMES, encoding='utf-8-sig')[['MSOA11CD', 'MSOA11NM']].drop_duplicates()
        .rename(columns={'MSOA11CD': 'msoa11cd', 'MSOA11NM': 'msoa_name'}))
df = df.merge(nm, on='msoa11cd', how='left')
assert df['msoa_name'].notna().all(), 'unnamed MSOAs after lookup merge'

# Ext arm share = Ext_Outflow_nat / Outflow_Poorer_nat, defined only above national D6
for yr in ('11', '21'):
    with np.errstate(divide='ignore', invalid='ignore'):
        s = df[f'Ext_Outflow_nat_{yr}'] / df[f'Outflow_Poorer_nat_{yr}']
    df[f'Ext_Arm_Share_{yr}'] = np.where(df['Wealth_Decile_National'] > 6, s.fillna(0), 0.0)

def leaf(r, yr):
    if r[f'Typ_C_{yr}'] != 'Cascade-led':
        return None
    if r[f'Casc_Inflow_Share_{yr}'] >= INFLOW_MIN:
        return 'inflow-driven' if r[f'Typ_A_{yr}'] == 'Cascade-led' else 'frame-sensitive'
    return 'outflow-external' if r[f'Ext_Arm_Share_{yr}'] >= EXT_MAJ else 'outflow-internal'

for yr in ('11', '21'):
    df[f'leaf_{yr}'] = df.apply(leaf, axis=1, yr=yr)
df['Genuine_Persistent'] = (df['leaf_11'] == 'inflow-driven') & (df['leaf_21'] == 'inflow-driven')

# guard: must reproduce the EDA 9 / EDA 12 leaf counts exactly
assert df['leaf_11'].value_counts().to_dict() == {'outflow-internal': 81, 'frame-sensitive': 57,
                                                  'inflow-driven': 56, 'outflow-external': 19}
assert df['leaf_21'].value_counts().to_dict() == {'outflow-internal': 90, 'outflow-external': 78,
                                                  'frame-sensitive': 14, 'inflow-driven': 13}
assert df['Genuine_Persistent'].sum() == 8
N_LONDON = len(df)
print('tree reproduces EDA 9 exactly — 2011:', df.leaf_11.value_counts().to_dict())
print('                              2021:', df.leaf_21.value_counts().to_dict(),
      '| genuine persistent n =', df.Genuine_Persistent.sum())
print('London MSOA sample N =', N_LONDON)

In [ ]:
# ── Yee labels: modal, any-GEN, Class-3 subtypes (identical to EDA 12 v3) ────
yee = pd.read_csv(YEE_LABELS).rename(columns={'LSOA_Code': 'lsoa11cd'})
lk  = pd.read_csv(LOOKUP)[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
yg  = yee.merge(lk, on='lsoa11cd', how='inner')
yg  = yg[yg['msoa11cd'].isin(df['msoa11cd'])].copy()
print(f'Yee LSOAs in frame: {len(yg)} across {yg.msoa11cd.nunique()}/{len(df)} MSOAs')

g = yg.groupby('msoa11cd')['Class_2_status']
modal = g.agg(lambda s: s.value_counts().index[0]).rename('yee_modal')
conf  = g.agg(lambda s: s.value_counts().iloc[0] / len(s)).rename('modal_confidence')
has_gen = (yg[yg['Class_2_status'] == 'GEN'].groupby('msoa11cd').size()
           .reindex(modal.index, fill_value=0) > 0).rename('has_gen')
df = df.merge(pd.concat([modal, conf, has_gen], axis=1),
              left_on='msoa11cd', right_index=True, how='left')
df['gen_modal'] = df['yee_modal'] == 'GEN'

for sub in ['SupGen', 'MainGen', 'MargGen']:
    df[f'any_{sub}'] = (yg[yg['Class_3_status'] == sub].groupby('msoa11cd').size()
                        .reindex(df['msoa11cd'], fill_value=0) > 0).values

print('modal GEN:', df.gen_modal.sum(), '| any-GEN:', df.has_gen.sum(),
      '| any-SupGen:', df.any_SupGen.sum(), '| any-MainGen:', df.any_MainGen.sum(),
      '| any-MargGen:', df.any_MargGen.sum())

# base rates in percent, for the "compared with Y% of all London MSOAs" clause
FLAGS = {'gen_modal': 'modal gentrification (strict)',
         'has_gen':   'any gentrification LSOA (loose)',
         'any_SupGen':'super-gentrification',
         'any_MainGen':'mainstream gentrification',
         'any_MargGen':'marginal gentrification'}
base_rates = {c: df[c].mean() * 100 for c in FLAGS}
print('\nLondon base rates (%):')
for c, v in base_rates.items():
    print(f'  {FLAGS[c]:<34} {v:5.1f}%  ({int(df[c].sum())}/{N_LONDON})')

---
## 1. Proportion machinery

For a group *g* (a mechanism leaf) and a label flag *L*:

- **`pct_group`** = 100 · P(*L* | *g*) — the headline number: *"L occurred in X % of g MSOAs"*.
- **`pct_london`** = 100 · P(*L*) — the comparator: *"compared with Y % of all London MSOAs"*.
- **`pp_diff`** = `pct_group` − `pct_london`, in percentage points — the plain-language effect size.
- **Wilson 95 % CI** on `pct_group`, which behaves correctly at the small leaf sizes here
  (n = 13–90) where a Wald interval would run outside [0, 1].
- Retained for the appendix only: **ratio** (= the EDA 12 "lift"), **odds ratio**, **Fisher exact p**
  (two-sided, exact at any n), **BH q** across the eight leaf tests per label family.

The Wilson interval is on the leaf proportion, not on the difference; it is there to stop a reader
treating `52.2 %` from n = 90 and `61.5 %` from n = 13 as equally precise. Where a chapter sentence needs
an interval on the *gap*, use the Newcombe interval printed alongside.

In [ ]:
# ── proportion machinery ─────────────────────────────────────────────────────
def wilson(k, n, z=1.96):
    "Wilson score interval for a binomial proportion; returns (lo, hi) in percent."
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half   = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - half) * 100, min(1.0, centre + half) * 100)


def newcombe(k1, n1, k2, n2, z=1.96):
    "Newcombe hybrid-score CI for the difference of two proportions, in percentage points."
    if n1 == 0 or n2 == 0:
        return (np.nan, np.nan)
    l1, u1 = [v / 100 for v in wilson(k1, n1, z)]
    l2, u2 = [v / 100 for v in wilson(k2, n2, z)]
    d = k1 / n1 - k2 / n2
    lo = d - np.sqrt((k1 / n1 - l1)**2 + (u2 - k2 / n2)**2)
    hi = d + np.sqrt((u1 - k1 / n1)**2 + (k2 / n2 - l2)**2)
    return (lo * 100, hi * 100)


def bh(pvals):
    "Benjamini-Hochberg step-up q-values."
    p = np.asarray(pvals, dtype=float); m = len(p); order = np.argsort(p)
    q = np.empty(m); prev = 1.0
    for rank, i in list(enumerate(order, 1))[::-1]:
        prev = min(prev, p[i] * m / rank); q[i] = prev
    return q


def prop_row(mask, flag, label):
    'One group x one label: proportions first, ratios kept for the appendix.'
    mask = np.asarray(mask, dtype=bool); flag = np.asarray(flag, dtype=bool)
    N, K = len(flag), int(flag.sum())                    # London sample, London positives
    n, k = int(mask.sum()), int((mask & flag).sum())     # leaf, leaf positives
    n_rest, k_rest = N - n, K - k                        # rest-of-London comparator
    pct_group  = 100 * k / n if n else np.nan
    pct_london = 100 * K / N
    pct_rest   = 100 * k_rest / n_rest if n_rest else np.nan
    ci_lo, ci_hi = wilson(k, n)
    d_lo, d_hi   = newcombe(k, n, k_rest, n_rest)
    OR, p = stats.fisher_exact([[k, n - k], [k_rest, n_rest - k_rest]], alternative='two-sided')
    return dict(group=label, n=n, n_label=k,
                pct_group=round(pct_group, 1), pct_london=round(pct_london, 1),
                pp_diff=round(pct_group - pct_london, 1),
                ci95_lo=round(ci_lo, 1), ci95_hi=round(ci_hi, 1),
                pct_rest=round(pct_rest, 1),
                pp_vs_rest=round(pct_group - pct_rest, 1),
                pp_vs_rest_lo=round(d_lo, 1), pp_vs_rest_hi=round(d_hi, 1),
                ratio=round((k / n) / (K / N), 2) if n and K else np.nan,
                OR=round(OR, 2), fisher_p=p)


def leaf_table(flag_col, label_name, corrected_rows=8):
    "Eight leaf tests (2 years x 4 leaves) for one label, BH-corrected within the family."
    rows = []
    for yr in ('11', '21'):
        for lf in LEAVES:
            r = prop_row(df[f'leaf_{yr}'] == lf, df[flag_col], lf)
            r['year'] = f'20{yr}'; r['label'] = label_name; r['flag_col'] = flag_col
            rows.append(r)
    t = pd.DataFrame(rows)
    t['bh_q'] = bh(t['fisher_p'].values[:corrected_rows]).tolist() + \
                [np.nan] * (len(t) - corrected_rows)
    return t[['label', 'year', 'group', 'n', 'n_label', 'pct_group', 'ci95_lo', 'ci95_hi',
              'pct_london', 'pp_diff', 'pct_rest', 'pp_vs_rest', 'pp_vs_rest_lo',
              'pp_vs_rest_hi', 'ratio', 'OR', 'fisher_p', 'bh_q', 'flag_col']]

print('machinery ready — Wilson + Newcombe intervals, Fisher retained for the appendix')

---
## 2. Main tables — proportions

These are the tables the chapter quotes from. Read a row as:

> *In [year], [label] occurred in **pct_group %** of [group] MSOAs, compared with **pct_london %** of all
> London MSOAs.*

`pct_rest` / `pp_vs_rest` give the strictly correct comparator for the Fisher test (leaf versus the rest of
London, not versus a whole that contains the leaf). The two differ only trivially at these leaf sizes, and
the chapter uses the whole-sample comparator because it is the one a reader can hold in their head — but
both are printed so the choice is visible and defensible.

In [ ]:
# ── main proportion tables: Class-3 subtypes, plus the two GEN aggregations ──
MAIN = pd.concat([leaf_table(c, FLAGS[c]) for c in
                  ['any_SupGen', 'any_MainGen', 'any_MargGen', 'gen_modal', 'has_gen']],
                 ignore_index=True)

show = ['year', 'group', 'n', 'n_label', 'pct_group', 'ci95_lo', 'ci95_hi',
        'pct_london', 'pp_diff', 'bh_q']
for c in ['any_SupGen', 'any_MainGen', 'any_MargGen', 'gen_modal', 'has_gen']:
    t = MAIN[MAIN.flag_col == c].copy()
    t['bh_q'] = t['bh_q'].map(lambda v: '' if pd.isna(v) else
                              (f'{v:.4f}' if v >= 1e-4 else f'{v:.1e}'))
    print(f'===== {FLAGS[c]} — {int(df[c].sum())}/{N_LONDON} London MSOAs '
          f'({base_rates[c]:.1f}%) =====')
    print(t[show].to_string(index=False))
    print()

### Reading the tables in proportion units

The substantive findings are unchanged from EDA 12 — only their expression is. Three things become
visible that lift concealed:

1. **The super-gentrification result is large in absolute terms, not just in ratio terms.** The
   outflow-internal leaf's share is roughly half of the leaf against roughly a fifth of London;
   the 2021 ratio of 3.37 corresponds to a gap of tens of percentage points, not a handful. Reporting
   the proportion strengthens the claim rather than softening it.
2. **The marginal-gentrification depletion is modest in absolute terms.** "Significantly depleted"
   read as a dramatic finding; the proportion shows a single-digit-to-low-double-digit percentage
   gap on a large leaf, which is exactly what a p-value of 0.018 on n = 78 should look like. This is
   the honest presentation.
3. **The small-n cells announce themselves.** The 2021 inflow-driven leaf (n = 13) and the
   genuine-persistent set (n = 8) carry Wilson intervals spanning tens of points; no reader can mistake
   `0.00` or `61.5 %` there for a precise estimate. In lift units the same cells looked like
   confident numbers.

The prose in §3 is generated directly from these tables, so the chapter cannot drift from the numbers.

In [ ]:
# ── chapter-ready sentences, generated from the tables ───────────────────────
LEAF_PROSE = {'inflow-driven':     'inflow-driven',
              'frame-sensitive':   'frame-sensitive',
              'outflow-external':  'outflow-external-majority',
              'outflow-internal':  'outflow-internal-majority'}
YEAR_ROLE  = {'2011': 'concurrent validation', '2021': 'decade-later persistence test'}
SUB_PROSE  = {'any_SupGen':  'Super-gentrification',
              'any_MainGen': 'Mainstream gentrification',
              'any_MargGen': 'Marginal gentrification',
              'gen_modal':   'Modal gentrification',
              'has_gen':     'Gentrification (any-LSOA definition)'}


def sentence(flag_col, year, leaf_name, with_ci=True, with_stats=False):
    r = MAIN[(MAIN.flag_col == flag_col) & (MAIN.year == year) &
             (MAIN.group == leaf_name)].iloc[0]
    more = 'more' if r.pp_diff > 0 else 'less'
    gap  = abs(r.pp_diff)
    unit = 'percentage point' if round(gap) == 1 else 'percentage points'
    s = (f"{SUB_PROSE[flag_col]} occurred in {r.pct_group:.0f}% of {year} "
         f"{LEAF_PROSE[leaf_name]} MSOAs ({int(r.n_label)} of {int(r.n)}), compared with "
         f"{r.pct_london:.0f}% of all London MSOAs — a gap of "
         f"{gap:.0f} {unit}, i.e. it was {more} frequent in this leaf "
         f"than in the London sample overall.")
    if with_ci:
        s += f" [95% CI on the leaf proportion: {r.ci95_lo:.0f}–{r.ci95_hi:.0f}%]"
    if with_stats:
        q = r.bh_q
        s += f" [ratio {r.ratio:.2f}; Fisher p = {r.fisher_p:.2g}; BH q = {q:.2g}]"
    return s


HIGHLIGHTS = [('any_SupGen',  '2011', 'outflow-internal'),
              ('any_SupGen',  '2021', 'outflow-internal'),
              ('any_MainGen', '2011', 'inflow-driven'),
              ('any_MainGen', '2021', 'inflow-driven'),
              ('any_MargGen', '2021', 'outflow-external'),
              ('gen_modal',   '2011', 'outflow-internal'),
              ('gen_modal',   '2021', 'outflow-internal'),
              ('has_gen',     '2021', 'outflow-external')]

print('CHAPTER SENTENCES (proportion-first, drop the bracketed CI if the line runs long)\n')
for fc, yr, lf in HIGHLIGHTS:
    print(f'· [{yr} — {YEAR_ROLE[yr]}] ' + sentence(fc, yr, lf))
    print()

In [ ]:
# ── the two-sentence pair the Results chapter needs verbatim ────────────────
sup11 = MAIN[(MAIN.flag_col == 'any_SupGen') & (MAIN.year == '2011') &
             (MAIN.group == 'outflow-internal')].iloc[0]
sup21 = MAIN[(MAIN.flag_col == 'any_SupGen') & (MAIN.year == '2021') &
             (MAIN.group == 'outflow-internal')].iloc[0]
marg21 = MAIN[(MAIN.flag_col == 'any_MargGen') & (MAIN.year == '2021') &
              (MAIN.group == 'outflow-external')].iloc[0]

# direction words are derived, never asserted: if a value moves, the prose moves with it
sup_dir   = 'larger' if (sup11.pp_diff > 0 and sup21.pp_diff > 0) else 'different'
sup_trend = ('strengthens over the decade' if sup21.pp_diff > sup11.pp_diff
             else 'weakens over the decade' if sup21.pp_diff < sup11.pp_diff
             else 'is stable across the decade')
marg_dir  = 'less' if marg21.pp_diff < 0 else 'more'

para = (
 f"Super-gentrification formed a {sup_dir} proportion of the outflow-internal-majority leaf than of the "
 f"London sample overall in both comparisons. It occurred in {sup11.pct_group:.0f}% of "
 f"outflow-internal-majority MSOAs in the concurrent test ({int(sup11.n_label)} of {int(sup11.n)}) and "
 f"{sup21.pct_group:.0f}% in the persistence test ({int(sup21.n_label)} of {int(sup21.n)}), compared with "
 f"{sup11.pct_london:.0f}% of all London MSOAs — gaps of {sup11.pp_diff:.0f} and {sup21.pp_diff:.0f} "
 f"percentage points respectively. The correspondence therefore holds concurrently and {sup_trend}. "
 f"Marginal gentrification, by contrast, occurred {marg_dir} frequently among 2021 "
 f"outflow-external-majority MSOAs ({marg21.pct_group:.0f}%, {int(marg21.n_label)} of {int(marg21.n)}) "
 f"than in the full London sample ({marg21.pct_london:.0f}%), a gap of {abs(marg21.pp_diff):.0f} "
 f"percentage points in the opposite direction. The corresponding ratios, odds ratios and exact-test "
 f"results are reported in Appendix Table A.x."
)
print(para)

---
## 3. Figure — proportions against the London base rate

Replaces `fig_eda12_lift_by_leaf.png` in the chapter. The dashed line is the London base rate for that
label; each bar is a leaf proportion with its Wilson 95 % interval. A reader sees the two quantities the
sentence names — the leaf percentage and the London percentage — rather than a ratio they must decompose.
Keep the EDA 12 lift dot plot for the appendix if the ratio view is still wanted there.

In [ ]:
# ── figure: leaf proportions vs London base rate ─────────────────────────────
plt.rcParams['font.family'] = 'DejaVu Sans'

panels = ['any_SupGen', 'any_MainGen', 'any_MargGen', 'gen_modal']
COL = {'2011': '#8073ac', '2021': '#e08214'}
fig, axes = plt.subplots(1, len(panels), figsize=(17, 4.9), sharey=True)
ypos = {lf: i for i, lf in enumerate(LEAVES[::-1])}

for ax, fc in zip(axes, panels):
    t = MAIN[MAIN.flag_col == fc]
    base = base_rates[fc]
    for _, r in t.iterrows():
        y = ypos[r.group] + (0.18 if r.year == '2011' else -0.18)
        ax.barh(y, r.pct_group, height=0.32, color=COL[r.year],
                alpha=0.95 if (pd.notna(r.bh_q) and r.bh_q < 0.05) else 0.45,
                edgecolor='black' if (pd.notna(r.bh_q) and r.bh_q < 0.05) else 'none', lw=0.9)
        ax.plot([r.ci95_lo, r.ci95_hi], [y, y], color='#333', lw=1.1, zorder=4)
        ax.annotate(f'{r.pct_group:.0f}%  ({int(r.n_label)}/{int(r.n)})',
                    (max(r.ci95_hi, r.pct_group), y), textcoords='offset points',
                    xytext=(5, -3), fontsize=8, color='#333')
    ax.axvline(base, color='grey', lw=1.4, ls='--')
    ax.annotate(f'London {base:.0f}%', (base, 3.62), fontsize=8, color='grey',
                rotation=90, ha='right', va='top')
    ax.set_yticks(range(4)); ax.set_yticklabels([LEAF_PROSE[l] for l in LEAVES[::-1]])
    ax.set_title(f'{FLAGS[fc]}\n({int(df[fc].sum())}/{N_LONDON} London MSOAs)', fontsize=10)
    ax.set_xlabel('% of MSOAs in leaf carrying the label')
    ax.set_xlim(0, 100)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)

handles = [plt.Rectangle((0, 0), 1, 1, color=COL[y], label=f'{y} — {YEAR_ROLE[y]}')
           for y in ('2011', '2021')]
handles += [plt.Rectangle((0, 0), 1, 1, fc='white', ec='black', label='BH q < 0.05 (outlined)'),
            plt.Line2D([], [], color='grey', ls='--', label='London base rate')]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, fontsize=9)
fig.suptitle('Yee label prevalence by mechanism leaf, against the London base rate '
             '(bars = leaf %, whiskers = Wilson 95% CI)', fontweight='bold')
fig.tight_layout(rect=[0, 0.08, 1, 0.99])
fig.savefig(FIG_DIR / 'fig_eda13_proportions_by_leaf.png', dpi=200, bbox_inches='tight')
plt.show()

---
## 4. Appendix table — ratios, odds ratios, exact tests

This is where lift now lives. The chapter cites it once, as *"the corresponding ratios and exact-test
results are reported in Appendix Table A.x"*, and never runs prose off it.

In [ ]:
# ── appendix table: ratio / OR / p / q ───────────────────────────────────────
APPX = MAIN[['label', 'year', 'group', 'n', 'n_label', 'pct_group', 'pct_london',
             'pp_diff', 'ratio', 'OR', 'fisher_p', 'bh_q']].copy()
fmt = lambda v: '' if pd.isna(v) else (f'{v:.4f}' if v >= 1e-4 else f'{v:.1e}')
APPX_disp = APPX.copy()
APPX_disp['fisher_p'] = APPX_disp['fisher_p'].map(fmt)
APPX_disp['bh_q'] = APPX_disp['bh_q'].map(fmt)
print(APPX_disp.to_string(index=False))

---
## 5. Consistency check against EDA 12

The ratios computed here must reproduce the lift values EDA 12 reported, or one of the two notebooks is
wrong. Checked against the values quoted in Results v3 §4.5.2–4.5.3.

In [ ]:
# ── consistency: ratios here == lift values in EDA 12 / Results v3 ──────────
EXPECTED = [   # (flag_col, year, leaf, lift reported in EDA 12 / Results v3)
    ('any_SupGen',  '2011', 'outflow-internal', 2.32),
    ('any_SupGen',  '2021', 'outflow-internal', 3.37),
    ('gen_modal',   '2011', 'outflow-internal', 3.55),
    ('gen_modal',   '2021', 'outflow-internal', 3.19),
    ('any_MainGen', '2011', 'inflow-driven',    2.25),
    ('has_gen',     '2021', 'outflow-external', 0.43),
]
ok = True
for fc, yr, lf, expected in EXPECTED:
    got = MAIN[(MAIN.flag_col == fc) & (MAIN.year == yr) & (MAIN.group == lf)].iloc[0].ratio
    flag = 'OK ' if abs(got - expected) <= 0.02 else 'MISMATCH'
    ok &= (flag == 'OK ')
    print(f'{flag} {FLAGS[fc]:<34} {yr} {lf:<17} ratio {got:>5.2f}  (EDA 12 lift {expected:.2f})')
assert ok, 'EDA 13 ratios do not reproduce EDA 12 lift — investigate before using either notebook'
print('\nall ratios reproduce EDA 12 — the two notebooks report the same tests in different units')

---
## 6. Exports

- `eda13_proportions_main_20260723.csv` — the chapter-facing proportion table.
- `eda13_appendix_ratios_20260723.csv` — Appendix Table A.x.
- `eda13_chapter_prose_20260723.md` — generated sentences, ready to paste into Results §4.5.2–4.5.3.
- `fig_eda13_proportions_by_leaf.png` — replacement for the lift dot plot in the main chapter.

In [ ]:
# ── exports ──────────────────────────────────────────────────────────────────
MAIN.to_csv(OUT_DIR / f'eda13_proportions_main_{STAMP}.csv', index=False)
APPX.to_csv(OUT_DIR / f'eda13_appendix_ratios_{STAMP}.csv', index=False)

lines = ['# EDA 13 — proportion-first sentences for Results §4.5', '',
         f'Generated {STAMP}. London sample N = {N_LONDON} MSOAs.', '',
         '## Replacement paragraph (§4.5.3, super-gentrification and the exodus)', '',
         para, '', '## Individual sentences', '']
for fc, yr, lf in HIGHLIGHTS:
    lines += [f'**{FLAGS[fc]} × {LEAF_PROSE[lf]}, {yr} ({YEAR_ROLE[yr]})**', '',
              sentence(fc, yr, lf, with_ci=True, with_stats=True), '']
lines += ['## Base rates', '']
lines += [f'- {FLAGS[c]}: {v:.1f}% of London MSOAs ({int(df[c].sum())}/{N_LONDON})'
          for c, v in base_rates.items()]

(OUT_DIR / f'eda13_chapter_prose_{STAMP}.md').write_text('\n'.join(lines), encoding='utf-8')
print('exported:',
      f'eda13_proportions_main_{STAMP}.csv', MAIN.shape, '|',
      f'eda13_appendix_ratios_{STAMP}.csv', APPX.shape, '|',
      f'eda13_chapter_prose_{STAMP}.md', '|',
      'fig_eda13_proportions_by_leaf.png')

---
## 7. What to change in the chapter

**§4.5.2 (leaf-level enrichment).** Rewrite the lead so the leaf and London percentages appear before any
ratio. Replace *"is GEN-depleted (loose lift 0.43, p = 0.0055)"* with the proportion pair and the phrase
*"occurred less frequently than in the London sample overall"*. Keep one parenthetical pointer to the
appendix rather than a p-value per clause.

**§4.5.3 (subtype decomposition).** Use the generated paragraph in `eda13_chapter_prose_20260723.md`. The
sentence *"lift 2.32 (q < 0.01) … strengthening to 3.37 (q < 10⁻⁹)"* becomes the two proportions plus the
percentage-point gaps; the ratios stay, in the appendix, where the strengthening-over-the-decade claim can
still be verified.

**Table 4.x.** Swap the lift column set for `pct_group / pct_london / pp_diff / n`, with `ratio / OR / q`
moved to Appendix A.

**Figures.** `fig_eda13_proportions_by_leaf.png` in the main chapter; keep `fig_eda12_lift_by_leaf.png` in
the appendix beside Table A.x if a ratio view is still wanted.

**What does *not* change.** The mechanism tree, the leaf definitions, the Yee aggregation, the tests, the
correction, the conclusions in EDA 12 §3, or the deck's Findings 1 framing. The ladder-ceiling caveat still
applies to the super-gentrification result and should still be stated wherever that result is reported:
top-decile areas cannot be inflow-led, so the displacement arm is the only observable cascade signature
available to them.